# Flan-T5 LoRA Training on PubMedQA

This notebook demonstrates:
1. Model loading and inspection
2. LoRA (Low-Rank Adaptation) implementation for efficient fine-tuning
3. Training on PubMedQA dataset
4. Evaluation and inference

**Environment**: Kaggle with T4 GPU

## 1. Setup & Imports

In [ ]:
# Install required packages (uncomment if needed on Kaggle)
# !pip install datasets transformers tqdm rouge_score

In [ ]:
import re
import math
from collections import defaultdict, Counter
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import DataCollatorForSeq2Seq

from rouge_score import rouge_scorer

In [ ]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Utility Functions

### 2.1 LoRA Implementation

In [ ]:
class LoRALinear(nn.Module):
    """LoRA wrapper for linear layers."""
    
    def __init__(self, base_linear, r=8, alpha=1.0):
        super().__init__()
        self.base = base_linear
        self.base.weight.requires_grad = False  # Freeze base

        in_dim = base_linear.in_features
        out_dim = base_linear.out_features

        # Get the device of the base layer
        device = base_linear.weight.device

        self.A = nn.Linear(in_dim, r, bias=False, device=device)
        self.B = nn.Linear(r, out_dim, bias=False, device=device)
        self.scaling = alpha / r

        # Initialize LoRA
        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        return self.base(x) + self.scaling * self.B(self.A(x))

    # Expose weight/bias to satisfy HF generate()
    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

### 2.2 Model Modification Functions

In [ ]:
def freeze_all_params(model):
    """Freeze all model parameters."""
    for p in model.parameters():
        p.requires_grad = False


def enable_ffn_training(model):
    """Enable training only for FFN (DenseReluDense) layers in both encoder and decoder."""
    for name, module in model.named_modules():
        if module.__class__.__name__ == "T5DenseGatedActDense":
            for p in module.parameters():
                p.requires_grad = True


def apply_lora_to_ffn(model, r=8, alpha=1.0):
    """Apply LoRA to FFN layers (T5DenseGatedActDense)."""
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            continue  # Only wrap higher-level FFN
        if module.__class__.__name__ == "T5DenseGatedActDense":
            module.wi_0 = LoRALinear(module.wi_0, r=r, alpha=alpha)
            module.wi_1 = LoRALinear(module.wi_1, r=r, alpha=alpha)
            module.wo = LoRALinear(module.wo, r=r, alpha=alpha)


def merge_lora_to_linear(model):
    """Merge LoRA weights back into base linear layers."""
    for name, module in model.named_modules():
        if isinstance(module, LoRALinear):
            new_linear = nn.Linear(
                module.base.in_features,
                module.base.out_features,
                bias=(module.base.bias is not None)
            ).to(module.base.weight.device)

            new_linear.weight.data = module.base.weight.data + (module.B.weight @ module.A.weight) * module.scaling

            if module.base.bias is not None:
                new_linear.bias.data = module.base.bias.data

            parent = module._modules
            for key, child in parent.items():
                if child is module:
                    parent[key] = new_linear
                    break


def enable_lora_forward(model):
    """Temporarily compute effective weights for generation."""
    for module in model.modules():
        if isinstance(module, LoRALinear):
            module._original_forward = module.forward
            module.forward = lambda x, m=module: m.base(x) + m.scaling * m.B(m.A(x))


def disable_lora_forward(model):
    """Restore original forward for training."""
    for module in model.modules():
        if isinstance(module, LoRALinear) and hasattr(module, "_original_forward"):
            module.forward = module._original_forward
            del module._original_forward

### 2.3 Parameter Counting Utilities

In [ ]:
def count_parameters(model):
    """Count total and trainable parameters."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def print_trainable_params(model):
    """Print names of trainable parameters."""
    for name, p in model.named_parameters():
        if p.requires_grad:
            print(name)


def print_param_stats(model):
    """Print parameter statistics by module."""
    param_stats = defaultdict(int)
    for name, param in model.named_parameters():
        param_stats[name.split('.')[0]] += param.numel()
    for k, v in param_stats.items():
        print(f"{k}: {v:,}")


def extract_explanation(text):
    """Extract explanation from generated text."""
    # Try to extract text after "Explanation:"
    match = re.search(r"Explanation:\s*(.+)", text, re.IGNORECASE | re.DOTALL)
    if match:
        return match.group(1).strip()
    # If no "Explanation:" found, return text after "Answer: yes/no/maybe."
    match = re.search(r"Answer:\s*(?:yes|no|maybe)[.,]?\s*(.+)", text, re.IGNORECASE | re.DOTALL)
    if match:
        return match.group(1).strip()
    return text


def compute_rouge_scores(predictions, references, scorer):
    """Compute average ROUGE scores for a list of predictions and references."""
    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []
    
    for pred, ref in zip(predictions, references):
        # Extract explanations
        pred_expl = extract_explanation(pred)
        ref_expl = extract_explanation(ref)
        
        scores = scorer.score(ref_expl, pred_expl)
        rouge1_scores.append(scores['rouge1'].fmeasure)
        rouge2_scores.append(scores['rouge2'].fmeasure)
        rougeL_scores.append(scores['rougeL'].fmeasure)
    
    return {
        'rouge1': sum(rouge1_scores) / len(rouge1_scores) if rouge1_scores else 0,
        'rouge2': sum(rouge2_scores) / len(rouge2_scores) if rouge2_scores else 0,
        'rougeL': sum(rougeL_scores) / len(rougeL_scores) if rougeL_scores else 0
    }

## 3. Model Loading & Inspection

In [ ]:
MODEL_NAME = "google/flan-t5-base"

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
).to(device)

print(f"Model loaded on: {device}")

In [ ]:
# Quick test of the base model
model.eval()

prompt = "What can you do"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=128)

print("Base model response:")
print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

### 3.1 Model Architecture Inspection

In [ ]:
print("Full Model Architecture:")
print(model)

In [ ]:
print("Encoder Block 0:")
print(model.encoder.block[0])

In [ ]:
print("Decoder Block 0:")
print(model.decoder.block[0])

In [ ]:
print("LM Head:")
print(model.lm_head)

In [ ]:
total_params, trainable_params = count_parameters(model)
frozen_params = total_params - trainable_params

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {frozen_params:,}")

In [ ]:
print("\nParameter statistics by module:")
print_param_stats(model)

In [ ]:
print("\nFFN and Attention layers:")
for name, param in model.named_parameters():
    if "DenseReluDense" in name:
        print("FFN:", name, param.numel())
    elif "SelfAttention" in name or "EncDecAttention" in name:
        print("ATTN:", name, param.numel())

## 4. Dataset Preparation (PubMedQA)

### 4.1 Load and Preprocess Dataset

In [ ]:
# Preprocessing parameters
max_input_length = 512
max_output_length = 128

In [ ]:
def preprocess_pubmedqa(example):
    """Preprocess PubMedQA examples into input-output format."""
    context = " ".join(example["context"]["contexts"])
    input_text = (
        f"Question: {example['question']} "
        f"Context: {context} "
        f"Instruction: Answer yes, no, or maybe. Then justify your answer."
    )
    target_text = (
        f"Answer: {example['final_decision']}. "
        f"Explanation: {example['long_answer']}"
    )
    short_answer = example['final_decision']

    return {
        "input": input_text,
        "output": target_text,
        "short_answer": short_answer
    }


def tokenize_for_t5(example):
    """Tokenize examples for T5 training."""
    input_enc = tokenizer(
        example["input"],
        truncation=True,
        padding="max_length",
        max_length=max_input_length,
    )

    target_enc = tokenizer(
        example["output"],
        truncation=True,
        padding="max_length",
        max_length=max_output_length,
    )

    labels = target_enc["input_ids"]
    labels = [l if l != tokenizer.pad_token_id else -100 for l in labels]

    return {
        "input_ids": input_enc["input_ids"],
        "attention_mask": input_enc["attention_mask"],
        "labels": labels,
    }


def collate_fn(batch):
    """Collate function for DataLoader."""
    input_ids = torch.tensor([item["input_ids"] for item in batch], dtype=torch.long)
    attention_mask = torch.tensor([item["attention_mask"] for item in batch], dtype=torch.long)
    labels = torch.tensor([item["labels"] for item in batch], dtype=torch.long)
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [ ]:
# Load dataset
dataset = load_dataset("pubmed_qa", "pqa_labeled")
print(dataset)

In [ ]:
print("Dataset features:")
print(dataset["train"].features)

In [ ]:
print("\nSample example:")
print(dataset["train"][0])

In [ ]:
print("\nLabel distribution:")
print(Counter(dataset["train"]["final_decision"]))

In [ ]:
# Preprocess dataset
dataset = dataset.map(
    preprocess_pubmedqa,
    remove_columns=dataset["train"].column_names
)
print("\nPreprocessed example:")
print(dataset["train"][0])

In [ ]:
# Train/validation split
split_datasets = dataset["train"].train_test_split(test_size=0.1, seed=42)

train_data = split_datasets["train"]
val_data = split_datasets["test"]

print(f"Train size: {len(train_data)}")
print(f"Val size: {len(val_data)}")

In [ ]:
# Store validation short answers for evaluation
val_short_answers = [ex["short_answer"] for ex in val_data]
val_raw_inputs = [ex["input"] for ex in val_data]
val_raw_outputs = [ex["output"] for ex in val_data]

# Remove short_answer column before tokenization
train_data = train_data.remove_columns("short_answer")
val_data = val_data.remove_columns("short_answer")

print(f"Train columns: {train_data.column_names}")
print(f"Val columns: {val_data.column_names}")

In [ ]:
# Tokenize datasets
train_dataset = train_data.map(tokenize_for_t5, remove_columns=train_data.column_names)
val_dataset = val_data.map(tokenize_for_t5, remove_columns=val_data.column_names)

print(f"Tokenized train: {train_dataset}")
print(f"Tokenized val: {val_dataset}")

In [ ]:
# Create DataLoaders
batch_size = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

In [ ]:
# Verify batch structure
sample_batch = next(iter(train_loader))
print("Sample batch keys:", sample_batch.keys())
print("Input IDs shape:", sample_batch["input_ids"].shape)
print("Labels shape:", sample_batch["labels"].shape)

## 5. Apply LoRA and Prepare for Training

In [ ]:
# Reload model fresh for LoRA training
model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
).to(device)

print(f"Fresh model loaded on: {device}")

In [ ]:
# LoRA hyperparameters
rank = 128
alpha = 256

# Freeze all parameters
freeze_all_params(model)

# Apply LoRA to FFN layers
apply_lora_to_ffn(model, r=rank, alpha=alpha)

# Make only LoRA parameters trainable
for name, param in model.named_parameters():
    if "A.weight" in name or "B.weight" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

print(f"LoRA applied with rank={rank}, alpha={alpha}")

In [ ]:
# Verify parameter counts
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = total - trainable

print(f"Total parameters: {total:,}")
print(f"Trainable parameters (LoRA): {trainable:,}")
print(f"Frozen parameters: {frozen:,}")
print(f"Trainable %: {100 * trainable / total:.4f}%")

## 6. Training Loop with Evaluation

In [ ]:
# Training hyperparameters
num_epochs = 3
learning_rate = 1e-3

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=learning_rate
)

print(f"Training for {num_epochs} epochs with lr={learning_rate}")

In [ ]:
# Initialize ROUGE scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

for epoch in range(num_epochs):
    # Training phase
    model.train()
    running_loss = 0.0
    loop = tqdm(enumerate(train_loader, 1), total=len(train_loader), desc=f"Epoch {epoch+1}")

    for step, batch in loop:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, model.parameters()),
            max_norm=1.0
        )
        
        optimizer.step()

        running_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} average loss: {avg_loss:.4f}")

    # Evaluation phase
    model.eval()
    enable_lora_forward(model)

    correct = 0
    total_eval = 0
    global_idx = 0
    all_generated_texts = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=128,
                do_sample=False
            )

            gen_texts = [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]
            all_generated_texts.extend(gen_texts)

            for text in gen_texts:
                match = re.search(r"Answer:\s*(yes|no|maybe)", text, re.IGNORECASE)
                pred = match.group(1).lower() if match else ""
                true_label = val_short_answers[global_idx].lower()
                
                if pred == true_label:
                    correct += 1
                total_eval += 1
                global_idx += 1

    # Short answer accuracy
    accuracy = correct / total_eval if total_eval > 0 else 0.0
    print(f"Epoch {epoch+1} short-answer accuracy: {accuracy:.4f}")
    
    # Long answer ROUGE scores
    rouge_scores = compute_rouge_scores(all_generated_texts, val_raw_outputs, scorer)
    print(f"Epoch {epoch+1} ROUGE-1: {rouge_scores['rouge1']:.4f}, ROUGE-2: {rouge_scores['rouge2']:.4f}, ROUGE-L: {rouge_scores['rougeL']:.4f}")

    disable_lora_forward(model)

## 7. Inference After Training

In [ ]:
# Enable LoRA for generation
enable_lora_forward(model)
model.eval()

generated_texts = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Generating on val set"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=128,
            do_sample=False
        )

        batch_texts = [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]
        generated_texts.extend(batch_texts)

# Compute final metrics
# Short answer accuracy
correct = 0
for i, text in enumerate(generated_texts):
    match = re.search(r"Answer:\s*(yes|no|maybe)", text, re.IGNORECASE)
    pred = match.group(1).lower() if match else ""
    if pred == val_short_answers[i].lower():
        correct += 1
final_accuracy = correct / len(generated_texts)

# Long answer ROUGE scores
final_rouge = compute_rouge_scores(generated_texts, val_raw_outputs, scorer)

print("\n" + "="*80)
print("Final Evaluation Metrics:")
print("="*80)
print(f"Short-answer Accuracy: {final_accuracy:.4f}")
print(f"ROUGE-1: {final_rouge['rouge1']:.4f}")
print(f"ROUGE-2: {final_rouge['rouge2']:.4f}")
print(f"ROUGE-L: {final_rouge['rougeL']:.4f}")

# Display sample outputs
print("\n" + "="*80)
print("Sample Generations:")
print("="*80)

for i in range(min(5, len(generated_texts))):
    # Compute individual ROUGE for this example
    individual_rouge = scorer.score(
        extract_explanation(val_raw_outputs[i]), 
        extract_explanation(generated_texts[i])
    )
    
    print(f"\nExample {i+1}:")
    print(f"Question + Context + Instruction:\n{val_raw_inputs[i][:200]}...\n")
    print(f"True Answer: {val_raw_outputs[i]}")
    print(f"Generated Answer:\n{generated_texts[i]}")
    print(f"\nROUGE-1: {individual_rouge['rouge1'].fmeasure:.4f}, ROUGE-2: {individual_rouge['rouge2'].fmeasure:.4f}, ROUGE-L: {individual_rouge['rougeL'].fmeasure:.4f}")
    print("-" * 80)

disable_lora_forward(model)

### 7.1 Custom Inference Examples

In [ ]:
enable_lora_forward(model)
model.eval()

# Test with custom medical questions
test_questions = [
    "Question: Does aspirin reduce heart attack risk? Instruction: Answer in 2 sentences.",
    "Question: Is water wet? Instruction: Answer in one sentence.",
    "Question: Is vitamin C a cure for cancer? Answer yes or no.",
    "Question: Does aspirin reduce fever? Answer yes or no."
]

print("Custom Inference Examples:")
print("="*80)

for question in test_questions:
    inputs = tokenizer(question, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=64)
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\nQ: {question}")
    print(f"A: {response}")
    print("-" * 40)

disable_lora_forward(model)

## 8. Merge LoRA Weights and Save (Optional)

In [ ]:
# Merge LoRA weights into base model
merge_lora_to_linear(model)
model.eval()

print("LoRA weights merged into base model.")

In [ ]:
# Final inference after merge
sample_input = "Question: Does aspirin reduce heart attack risk? Instruction: Answer in 2 sentences."
inputs = tokenizer(sample_input, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=64)

print("Final inference after LoRA merge:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
# Optionally save the merged model
# model.save_pretrained("./flan-t5-base-pubmedqa-lora-merged")
# tokenizer.save_pretrained("./flan-t5-base-pubmedqa-lora-merged")
# print("Model saved!")